# AG_PRAXIS NB01 — Dataset Inventory

Everything else in this project rests on what is actually inside these files, so this
notebook reads them before anything gets built. The dataset paper describes the features
and the attacks, but a description is not the data, and I would rather meet a surprise
now than halfway through a training run.

The questions worth asking first are the ones that close options off. Is there a clock in
the data. Is there anything naming a sender or a receiver. Do all seventy-two files carry
the same columns. How are the recordings organised, and how many are there per attack.
How many rows does each attack have, and how far apart are the largest and the smallest.
And is there any column that sits still inside a recording while moving between
recordings, which would let a model tell the files apart rather than the attacks.

Nothing is trained here. The notebook writes two files. `dataset_inventory.json` records
every answer, so no later notebook has to re-derive them or, worse, assume them.
`config/feature_families.yaml` is a first grouping of the columns, written as a draft
because it comes from spelling and nothing else.

The data sits on Drive and the code sits in the repository, so the first block mounts one
and clones the other. It also records the commit it is running from. Every number printed
below belongs to that commit, and if the working tree is dirty I would rather see it now
than when I am trying to explain a result later.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Paths and the seed come from `config/base.yaml` rather than from anything typed into a
cell, so changing a path is a commit and not an edit I forget I made. Everything this
notebook writes goes to the artefacts folder on Drive, because the cloned repository
disappears when the session ends and anything written into it goes with it.

`FAST` is on by default, and here it controls one thing only: how much of each file gets
read for the column scan in step seven. Row counts are exact either way, since counting
newlines is cheap enough to do on everything. A run entered in the ledger needs `FAST=0`,
because the constancy check is worth nothing unless it has seen every row.

In [ ]:
import json
import random
import time

import numpy as np
import pandas as pd

from src import captures as cap
from src import inventory as inv

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
TRAIN_DIR = Path(CFG["paths"]["train_dir"])
TEST_DIR = Path(CFG["paths"]["test_dir"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])
OUT_DIR = ARTIFACTS / "NB01"

FAST = os.environ.get("FAST", "1") == "1"
SCAN_FRAC = CFG["fast_mode"]["subsample_frac"] if FAST else 1.0
SCAN_FULL = not FAST

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

print(f"seed       : {SEED}")
print(f"train dir  : {TRAIN_DIR}   exists={TRAIN_DIR.exists()}")
print(f"test dir   : {TEST_DIR}   exists={TEST_DIR.exists()}")
print(f"output dir : {OUT_DIR}")
print()
print(f"FAST       : {int(FAST)}")
if FAST:
    print(f"             the column scan reads {SCAN_FRAC:.0%} of each file, which makes it a")
    print("             screen and not a finding. Rerun with FAST=0 before the ledger.")
else:
    print("             the column scan reads every row of every file.")

No model is built here, but the seed is set before anything else runs so that any
sampling I do while looking at the data gives me the same rows the next time.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
print(f"seeded with {SEED}")

The first real step is to read one file all the way through and look at every column: the
type it came back as, how many values are missing, how many distinct values it takes, and
where the numbers run from and to.

The distinct count is the part I watch hardest. A column holding one value carries no
information at all. A column with as many distinct values as there are rows is an
identifier rather than a measurement, which would be a different kind of problem and a
worse one.

In [ ]:
train_files = sorted(TRAIN_DIR.glob("*.csv"))
test_files = sorted(TEST_DIR.glob("*.csv"))
ALL_FILES = train_files + test_files
EXPECTED_FILES = 72

print(f"train files : {len(train_files)}")
print(f"test files  : {len(test_files)}")
print(f"total       : {len(ALL_FILES)}   (expected {EXPECTED_FILES})")
if len(ALL_FILES) != EXPECTED_FILES:
    print(f"MISMATCH: found {len(ALL_FILES)}, not {EXPECTED_FILES}. Check the paths above.")
print()

REFERENCE_FILE = train_files[0]
ref = pd.read_csv(REFERENCE_FILE)

print(f"reference file : {REFERENCE_FILE.name}")
print(f"shape          : {ref.shape[0]:,} rows x {ref.shape[1]} columns")
print()

schema_table = inv.describe_columns(ref)
print(schema_table.to_string(index=False))

REFERENCE_COLUMNS = list(ref.columns)
REFERENCE_DTYPES = {c: str(t) for c, t in ref.dtypes.items()}

single_valued = schema_table.loc[schema_table["n_unique"] <= 1, "column"].tolist()
row_unique = schema_table.loc[schema_table["n_unique"] == len(ref), "column"].tolist()
with_nulls = schema_table.loc[schema_table["nulls"] > 0, "column"].tolist()

print()
print(f"single-valued in this file : {single_valued or 'none'}")
print(f"distinct once per row      : {row_unique or 'none'}")
print(f"holding any nulls          : {with_nulls or 'none'}")

Now the first of the two questions that decide what can be built at all. Is there a clock
in the data.

I search the column names for time, timestamp, ts, date and epoch, and I do it two ways.
A whole-word match is a real hit. A letters-anywhere match is not, because `ts` sits
inside plenty of ordinary words, so those are printed separately rather than counted as
findings.

In [ ]:
time_hits = inv.search_columns(REFERENCE_COLUMNS, inv.TIME_KEYWORDS)
HAS_TIMESTAMP = bool(time_hits["word_hits"])

print(f"searched {len(REFERENCE_COLUMNS)} column names for: {', '.join(inv.TIME_KEYWORDS)}")
print()
for col, kws in time_hits["word_hits"].items():
    print(f"  whole word   : {col}   [{', '.join(kws)}]")
for col, kws in time_hits["substring_hits"].items():
    print(f"  letters only : {col}   [{', '.join(kws)}]   coincidence, not a field")
if not time_hits["word_hits"] and not time_hits["substring_hits"]:
    print("  nothing matched, on either reading")
print()
print("ANSWER   timestamp column present:", "YES" if HAS_TIMESTAMP else "NO")

The second question is whether anything identifies an endpoint. The same search with
different words: src, dst, ip, mac, addr, source, destination and port. This one decides
whether records can be grouped into flows or into a communication graph at all, or
whether the files have already been reduced to per-record statistics with the addressing
thrown away.

In [ ]:
endpoint_hits = inv.search_columns(REFERENCE_COLUMNS, inv.ENDPOINT_KEYWORDS)
HAS_ENDPOINTS = bool(endpoint_hits["word_hits"])

print(f"searched {len(REFERENCE_COLUMNS)} column names for: {', '.join(inv.ENDPOINT_KEYWORDS)}")
print()
for col, kws in endpoint_hits["word_hits"].items():
    print(f"  whole word   : {col}   [{', '.join(kws)}]")
for col, kws in endpoint_hits["substring_hits"].items():
    print(f"  letters only : {col}   [{', '.join(kws)}]   coincidence, not a field")
if not endpoint_hits["word_hits"] and not endpoint_hits["substring_hits"]:
    print("  nothing matched, on either reading")
print()
print("ANSWER   endpoint identifiers present:", "YES" if HAS_ENDPOINTS else "NO")

Those two answers together fix what a sequence can even be in this project, so the next
block spells that out rather than leaving me to remember it later. The point of writing it
down here is that both answers are structural. They are not something a better model or a
cleverer feature can work around.

In [ ]:
print("What the two answers mean for what can be built")
print()

if HAS_TIMESTAMP:
    print("A clock exists. Records can be sorted by time, the gap between two records is")
    print("measurable, and a window can be cut on elapsed time rather than on row position.")
    print("Sort order then has to be set explicitly and never assumed.")
else:
    print("There is no clock. The only ordering available is the order the rows appear in")
    print("the file, which is the order the feature extractor wrote them. That order means")
    print("something inside a file and nothing across files, so a window must never span")
    print("two files, and the row index must never be shuffled before windowing.")

print()

if HAS_ENDPOINTS:
    print("Endpoint identifiers exist. Records can be grouped per host or per flow before")
    print("windowing, and there are nodes and edges available for a communication graph.")
    print("They are also the most obvious shortcut in the data, so any model reading them")
    print("has to be checked for learning the address instead of the behaviour.")
else:
    print("Nothing identifies a sender or a receiver. Records cannot be grouped into flows")
    print("or per-host streams, and there are no nodes from which to build a communication")
    print("graph, so graph modelling is off the table for this dataset. A sequence here is")
    print("a run of consecutive rows from one file and nothing finer than that.")

print()
if not HAS_TIMESTAMP and not HAS_ENDPOINTS:
    window, stride = CFG["sequence"]["window"], CFG["sequence"]["stride"]
    print("Both answers are negative, which fixes the definition used from here on:")
    print(f"a sequence is {window} consecutive rows from a single file, taken every")
    print(f"{stride} rows, and never crossing a file boundary.")

One file agreeing with itself proves nothing about the other seventy-one. Reading only the
header costs almost nothing, so I read the header of every file and compare its column
list against the reference exactly, order included. Any file that differs gets printed
with what it is missing and what it has extra.

In [ ]:
column_differences = []

for path in ALL_FILES:
    cols = inv.header_of(path)
    if cols == REFERENCE_COLUMNS:
        continue
    missing = [c for c in REFERENCE_COLUMNS if c not in cols]
    extra = [c for c in cols if c not in REFERENCE_COLUMNS]
    column_differences.append(
        {
            "file": path.name,
            "n_columns": len(cols),
            "missing": missing,
            "extra": extra,
            "same_columns_reordered": not missing and not extra,
        }
    )

SCHEMA_IDENTICAL = not column_differences

print(f"reference : {REFERENCE_FILE.name}, {len(REFERENCE_COLUMNS)} columns")
print(f"compared  : {len(ALL_FILES)} files")
print()
if SCHEMA_IDENTICAL:
    print(f"ANSWER   all {len(ALL_FILES)} files carry the same {len(REFERENCE_COLUMNS)} columns")
    print("         in the same order. One column list holds for the whole dataset.")
else:
    print(f"ANSWER   {len(column_differences)} file(s) differ from the reference:")
    for d in column_differences:
        print(f"  {d['file']}   ({d['n_columns']} columns)")
        if d["missing"]:
            print(f"      missing : {d['missing']}")
        if d["extra"]:
            print(f"      extra   : {d['extra']}")
        if d["same_columns_reordered"]:
            print("      same columns, different order")

Next, how the recordings are organised, which is readable from the file names. A name like
`TCP_IP-DDoS-ICMP1_train.pcap.csv` says three things at once: the class is DDoS-ICMP, the
recording is number one of that class's sessions, and this file is its training part.
Stripping the trailing digits gives the class, stripping the suffix gives the recording,
and `ARP_Spoofing` is the file name spelling of the class the paper calls Spoofing.

That rule lives in `src/captures.py` as `parse_capture` and is imported rather than
retyped, so every notebook after this one groups the files the same way. If the rule turns
out to be wrong, I want it wrong in one place.

The table below is the whole map: for each class, which recordings appear in the training
partition, which appear in the test partition, and how many of each. I check the class
count against nineteen because that is what the dataset is documented to contain, and a
disagreement means my parsing rule is wrong rather than the paper.

In [ ]:
parsed = pd.DataFrame([{"file": p.name, **cap.parse_capture(p.name)} for p in ALL_FILES])
parsed["path"] = [str(p) for p in ALL_FILES]

print("how the rule reads a few of the names:")
print(parsed[["file", "capture_id", "label", "partition"]].head(6).to_string(index=False))
print()


def capture_list(group, partition):
    return sorted(group.loc[group["partition"] == partition, "capture_id"].unique())


rows = []
for label, group in parsed.groupby("label"):
    train_caps = capture_list(group, "train")
    test_caps = capture_list(group, "test")
    rows.append(
        {
            "label": label,
            "n_train_captures": len(train_caps),
            "n_test_captures": len(test_caps),
            "captures_train": train_caps,
            "captures_test": test_caps,
        }
    )

classes = pd.DataFrame(rows).sort_values("label").reset_index(drop=True)

printable = classes.assign(
    captures_train=classes["captures_train"].map(", ".join),
    captures_test=classes["captures_test"].map(", ".join),
)
print(printable.to_string(index=False))
print()

N_CLASSES = len(classes)
N_CAPTURES = int(parsed["capture_id"].nunique())

print(f"distinct classes    : {N_CLASSES}   (expected 19)")
print(f"distinct recordings : {N_CAPTURES}")
print(f"train files         : {int((parsed['partition'] == 'train').sum())}")
print(f"test files          : {int((parsed['partition'] == 'test').sum())}")

assert N_CLASSES == 19, f"expected 19 classes, found {N_CLASSES}: {sorted(classes['label'])}"
print()
print("19 classes confirmed.")

The number of training recordings a class has decides what kind of split is even possible
for it, so the tier comes straight off that table.

If a class was recorded more than once in the training partition, a whole recording can be
held out and the model tested on a session it has never seen. Call that tier A. If it was
recorded once, there is nothing to hold out, and the only split available is a cut inside
that single recording, where the training rows and the test rows come from the same
session minutes apart. Call that tier B.

Those are two different experiments and their scores are not comparable, which is why the
classes are labelled rather than pooled. I check the counts against eight and eleven for
the same reason as the nineteen above: if the split of classes across the two tiers comes
out differently, my reading of the file names is wrong somewhere.

In [ ]:
classes["tier"] = np.where(classes["n_train_captures"] > 1, "A", "B")

tier_a = classes.loc[classes["tier"] == "A", "label"].tolist()
tier_b = classes.loc[classes["tier"] == "B", "label"].tolist()

print("Tier A, more than one training recording, whole recordings can be held out")
for label in tier_a:
    n = int(classes.loc[classes["label"] == label, "n_train_captures"].iloc[0])
    print(f"  {label:<28} {n} training recordings")
print()
print("Tier B, one training recording, only a split inside that recording is possible")
for label in tier_b:
    print(f"  {label}")
print()
print(f"tier A : {len(tier_a):>2}   (expected 8)")
print(f"tier B : {len(tier_b):>2}   (expected 11)")

if (len(tier_a), len(tier_b)) != (8, 11):
    print()
    print(f"DIFFERS FROM EXPECTATION: {len(tier_a)} in A and {len(tier_b)} in B.")
    print("Tier A:", tier_a)
    print("Tier B:", tier_b)

assert (len(tier_a), len(tier_b)) == (8, 11), (
    f"expected 8 in tier A and 11 in tier B, got {len(tier_a)} and {len(tier_b)}"
)
print()
print("Tier split confirmed at 8 and 11.")

Sizes next. Parsing seventy-two files just to count their lines is wasteful, so I count
newlines instead and check that shortcut against pandas on one file before trusting it on
the rest. These counts are exact whatever `FAST` is set to.

The gap between the largest and the smallest class is the number that decides how results
get reported, so I take it three times, once for each grouping the benchmark is scored at.
Collapsing nineteen classes into six and then into two moves that gap a long way, and a
score that looks fine at two classes can hide a class that is never predicted at all.

In [ ]:
probe_fast, probe_pandas = inv.verify_row_count(REFERENCE_FILE)
print(f"newline count vs pandas on {REFERENCE_FILE.name} : {probe_fast:,} vs {probe_pandas:,}")
if probe_fast == probe_pandas:
    print("They agree, so newline counting is safe for the remaining files.")
else:
    print("They disagree, which means a field contains a newline. The counts below are wrong.")
print()

ROW_COUNTS = {p.name: inv.count_data_rows(p) for p in ALL_FILES}
parsed["rows"] = [ROW_COUNTS[p.name] for p in ALL_FILES]
TOTAL_ROWS = int(parsed["rows"].sum())

print("rows per file")
print(
    parsed[["file", "label", "capture_id", "partition", "rows"]]
    .sort_values(["label", "partition", "capture_id"])
    .to_string(index=False)
)
print()

class_sizes = (
    parsed.groupby("label")["rows"].sum().rename("rows").reset_index().merge(classes, on="label")
)
class_sizes["pct_of_total"] = (100 * class_sizes["rows"] / TOTAL_ROWS).round(3)
class_sizes = class_sizes.sort_values("rows", ascending=False).reset_index(drop=True)

print("rows per class, largest first")
print(
    class_sizes[["label", "tier", "rows", "pct_of_total", "n_train_captures", "n_test_captures"]]
    .to_string(index=False)
)
print()
print(f"total rows across {len(ALL_FILES)} files : {TOTAL_ROWS:,}")

In [ ]:
class_sizes["group6"] = class_sizes["label"].map(cap.group_six)
class_sizes["group2"] = class_sizes["label"].map(cap.group_two)

unmapped = class_sizes.loc[class_sizes["group6"] == "UNMAPPED", "label"].tolist()
assert not unmapped, f"labels the 6-class map does not cover: {unmapped}"


def grouped(column):
    frame = (
        class_sizes.groupby(column)
        .agg(classes=("label", "count"), rows=("rows", "sum"))
        .sort_values("rows", ascending=False)
    )
    frame["pct_of_total"] = (100 * frame["rows"] / TOTAL_ROWS).round(3)
    return frame


six, two = grouped("group6"), grouped("group2")

largest, smallest = class_sizes.iloc[0], class_sizes.iloc[-1]
RATIO_19 = float(largest["rows"]) / max(float(smallest["rows"]), 1.0)
RATIO_6 = float(six["rows"].max()) / max(float(six["rows"].min()), 1.0)
RATIO_2 = float(two["rows"].max()) / max(float(two["rows"].min()), 1.0)

print("6-class grouping")
print(six.to_string())
print()
print("2-class grouping")
print(two.to_string())
print()
print("largest class to smallest class")
print(f"  19-class : {RATIO_19:>10,.1f} to 1    {largest['label']} vs {smallest['label']}")
print(f"   6-class : {RATIO_6:>10,.1f} to 1    {six.index[0]} vs {six.index[-1]}")
print(f"   2-class : {RATIO_2:>10,.1f} to 1    {two.index[0]} vs {two.index[-1]}")
print()

share = float(largest["rows"]) / TOTAL_ROWS
majority_macro = (2 * share / (1 + share)) / 19
print(f"A model that always answers {largest['label']} and nothing else would be right")
print(f"{100 * share:.1f}% of the time and score a macro-F1 of {majority_macro:.3f} across the")
print("nineteen classes. The distance between those two numbers is why macro-F1 is the")
print("primary metric here and why accuracy is never reported on its own.")

The last check is the one I most want to come back negative, and it is looking for two
different things.

The first is a column that never changes anywhere in the dataset. A column like that
separates nothing from nothing, so it can be dropped without losing anything. Worth
knowing about, but harmless.

The second is the one that matters. If a column holds a single value for the whole of one
recording and a different single value in the next recording, then it is not describing
the traffic at all, it is naming the session. Each attack class here was recorded in its
own session, so a column that names the session also names the class. A model reading it
would score well on this dataset while having learned nothing about attacks, and the score
would collapse the moment it met traffic recorded anywhere else. That is the failure I
want to rule out before I trust any number this project produces.

I take a single file as the unit of a recording, because a file is one continuous stretch
of capture whereas a group of files is only a recording if I have grouped it correctly.
Afterwards I repeat the count with the files sharing a recording name treated as one, to
check the answer does not depend on that choice.

This has to see every row. A column that holds still through the first one percent of a
file and moves through the rest would be flagged as constant by a sample, and the flag
would be wrong. So the cell prints how much it actually read, and if that is not
everything then what follows is a screen and not a finding.

In [ ]:
started = time.time()
print(f"reading {len(ALL_FILES)} files at fraction {SCAN_FRAC:g}")
scans = cap.scan_captures(ALL_FILES, frac=SCAN_FRAC, known_rows=ROW_COUNTS)
ROWS_SCANNED = int(sum(s["n_rows_read"] for s in scans.values()))
print(f"done in {time.time() - started:.0f}s")
print()
print(f"rows read : {ROWS_SCANNED:,} of {TOTAL_ROWS:,}  ({100 * ROWS_SCANNED / TOTAL_ROWS:.1f}%)")
if SCAN_FULL:
    print("Every row was read, so what follows counts.")
else:
    print("FAST=1. This is a screen over the head of each file, not a finding. Rerun FAST=0.")
print()

# Two planted controls, one of each kind, to show the check can find them.
DEAD = "__planted_dead_column__"
MARKER = "__planted_recording_marker__"
planted = {
    name: {
        **scan,
        "columns": {
            **scan["columns"],
            DEAD: {"n_unique": 1, "value": 0.0},
            MARKER: {"n_unique": 1, "value": i},
        },
    }
    for i, (name, scan) in enumerate(scans.items())
}
control = cap.constancy_report(planted).set_index("column")
dead_ok = bool(control.loc[DEAD, "constant_everywhere"])
marker_ok = bool(control.loc[MARKER, "recording_identifying"])
print(f"control, one value everywhere      : classified constant everywhere = {dead_ok}")
print(f"control, one value per recording   : classified recording-naming    = {marker_ok}")
assert dead_ok and marker_ok, "the constancy check failed on its own planted controls"
print("Both controls land where they should, so the check is working.")
print()

SHOW = [
    "column",
    "constant_in_recordings",
    "of_recordings",
    "distinct_values_across_recordings",
]
report = cap.constancy_report(scans)

DEAD_COLUMNS = report.loc[report["constant_everywhere"], "column"].tolist()
IDENTIFYING = report.loc[report["recording_identifying"], "column"].tolist()
near_miss = report[
    (report["constant_in_recordings"] > 0)
    & ~report["constant_everywhere"]
    & ~report["recording_identifying"]
]

print("=" * 79)
print("FINDING 1   columns that never change anywhere in the dataset")
print("=" * 79)
if DEAD_COLUMNS:
    print(report[report["constant_everywhere"]][SHOW + ["values"]].to_string(index=False))
    print()
    print(f"{len(DEAD_COLUMNS)} column(s) hold one value across all {len(scans)} files.")
    print("They separate nothing and can be dropped.")
else:
    print("None. Every column takes more than one value somewhere.")
print()

print("=" * 79)
print("FINDING 2   columns constant inside a recording but different between recordings")
print("=" * 79)
if IDENTIFYING:
    print(report[report["recording_identifying"]][SHOW + ["values"]].to_string(index=False))
    print()
    print(f"{len(IDENTIFYING)} column(s) name the recording rather than measure the traffic.")
    print("They must be dropped before any model is fitted, and any earlier score that")
    print("used them means nothing.")
else:
    print("None. No column sits still for a whole recording and then moves to a new value")
    print("in the next one, so no single column acts as a name for the session a row came")
    print("from. That does not clear the dataset. It rules out the crudest version of the")
    print("problem, where one column is the giveaway. A shift in the distribution of a")
    print("column that varies within every recording would carry the same information more")
    print("quietly, and nothing here would catch it.")
print()

print("near misses, constant in some recordings but not all")
if len(near_miss):
    print(near_miss[SHOW].to_string(index=False))
else:
    print("  none")
print()

print("every column, ordered by how often it held still")
print(report[SHOW + ["constant_everywhere", "recording_identifying"]].to_string(index=False))

The count above treats each file as its own recording. Two files carrying the same
recording name are meant to be two parts of one session, and if that is what they are then
the answer should not change when they are counted together. If it does change, the
grouping is telling me something about the files that the names do not.

In [ ]:
capture_of = dict(zip(parsed["file"], parsed["capture_id"]))
by_capture = cap.constancy_report(scans, capture_of)

grouped_dead = by_capture.loc[by_capture["constant_everywhere"], "column"].tolist()
grouped_identifying = by_capture.loc[by_capture["recording_identifying"], "column"].tolist()

print(f"unit = one file          : {len(scans)} recordings, "
      f"{len(DEAD_COLUMNS)} dead, {len(IDENTIFYING)} recording-naming")
print(f"unit = one recording name: {parsed['capture_id'].nunique()} recordings, "
      f"{len(grouped_dead)} dead, {len(grouped_identifying)} recording-naming")
print()

GROUPING_AGREES = (set(grouped_dead) == set(DEAD_COLUMNS)) and (
    set(grouped_identifying) == set(IDENTIFYING)
)
if GROUPING_AGREES:
    print("The two readings agree, so the finding does not depend on how the files were")
    print("grouped into recordings.")
else:
    print("The two readings disagree:")
    print(f"  dead only when grouped      : {sorted(set(grouped_dead) - set(DEAD_COLUMNS))}")
    print(f"  dead only when ungrouped    : {sorted(set(DEAD_COLUMNS) - set(grouped_dead))}")
    print(f"  naming only when grouped    : {sorted(set(grouped_identifying) - set(IDENTIFYING))}")
    print(f"  naming only when ungrouped  : {sorted(set(IDENTIFYING) - set(grouped_identifying))}")

Two pictures, because the two numbers I will keep coming back to are hard to hold in a
table of nineteen rows.

The first is class size. It has to be on a log scale, because that is the only scale on
which classes spanning three orders of magnitude fit on one axis at all, and the fact that
a linear axis is unusable is itself the point. The second is how many recordings each
class has, which is where the tier comes from.

Both are coloured by tier, so the two figures can be read against each other: the classes
with the fewest rows are also the ones with the fewest recordings, and those are the same
classes that will be hardest to split honestly.

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, GRID, RULE = "#0b0b0b", "#52514e", "#e6e5e1", "#c9c8c3"
TIER_COLOUR = {"A": "#2a78d6", "B": "#eb6834"}
TIER_LABEL = {
    "A": "Tier A, more than one training recording",
    "B": "Tier B, one training recording",
}


def style(ax):
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(RULE)
    ax.tick_params(colors=MUTED, length=0)
    for label in ax.get_yticklabels():
        label.set_color(INK)


def tier_legend(ax):
    handles = [
        plt.Rectangle((0, 0), 1, 1, color=TIER_COLOUR[t], label=TIER_LABEL[t]) for t in ("A", "B")
    ]
    legend = ax.legend(handles=handles, loc="lower right", frameon=False, fontsize=9)
    for text in legend.get_texts():
        text.set_color(MUTED)


plot_order = class_sizes.sort_values("rows")

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(
    plot_order["label"],
    plot_order["rows"],
    color=[TIER_COLOUR[t] for t in plot_order["tier"]],
    height=0.68,
)
ax.set_xscale("log")
ax.set_xlim(right=plot_order["rows"].max() * 4)
ax.bar_label(
    bars,
    labels=[f"{int(v):,}" for v in plot_order["rows"]],
    padding=4,
    color=MUTED,
    fontsize=8,
)
ax.set_xlabel("rows, log scale", color=MUTED, fontsize=9)
ax.set_title(
    f"Rows per class, {TOTAL_ROWS:,} rows over 19 classes, "
    f"{RATIO_19:,.0f} to 1 largest to smallest",
    color=INK,
    fontsize=11,
    loc="left",
    pad=12,
)
style(ax)
tier_legend(ax)

FIG_SIZES = OUT_DIR / "NB01_class_sizes.png"
fig.savefig(FIG_SIZES, dpi=300, bbox_inches="tight", facecolor="white")
print(f"wrote {FIG_SIZES}")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(
    plot_order["label"],
    plot_order["n_train_captures"],
    color=[TIER_COLOUR[t] for t in plot_order["tier"]],
    height=0.68,
)
ax.bar_label(
    bars,
    labels=[
        f"{int(t)} train, {int(s)} test"
        for t, s in zip(plot_order["n_train_captures"], plot_order["n_test_captures"])
    ],
    padding=4,
    color=MUTED,
    fontsize=8,
)
ax.set_xlim(right=plot_order["n_train_captures"].max() + 2.6)
ax.set_xlabel("distinct training recordings", color=MUTED, fontsize=9)
ax.set_title(
    "Recordings per class, the count the tier is read from",
    color=INK,
    fontsize=11,
    loc="left",
    pad=12,
)
style(ax)
tier_legend(ax)

FIG_RECORDINGS = OUT_DIR / "NB01_recordings_per_class.png"
fig.savefig(FIG_RECORDINGS, dpi=300, bbox_inches="tight", facecolor="white")
print(f"wrote {FIG_RECORDINGS}")
plt.show()

Everything above now goes into one file. The point of writing it down is that no later
notebook should have to reread seventy-two files to find out how many classes there are,
and none of them should be allowed to assume it either. The file records the column list,
both answers about timestamps and identifiers with the searches that produced them, the
map from class to recording, the tiers, the row counts, and both constancy findings.

It also records whether it was written under `FAST`, because a screen and a finding should
never be mistaken for each other later.

In [ ]:
inventory_doc = {
    "generated_by": "AG_PRAXIS_NB01_dataset_inventory.ipynb",
    "generated_on": RUN_DATE,
    "git_sha": GIT_SHA,
    "git_dirty": GIT_DIRTY,
    "seed": SEED,
    "fast_mode": FAST,
    "scan_fraction": SCAN_FRAC,
    "scan_covered_every_row": SCAN_FULL,
    "rows_scanned": ROWS_SCANNED,
    "reference_file": REFERENCE_FILE.name,
    "n_files": len(ALL_FILES),
    "n_files_expected": EXPECTED_FILES,
    "n_columns": len(REFERENCE_COLUMNS),
    "n_classes": int(N_CLASSES),
    "n_captures": int(N_CAPTURES),
    "total_rows": TOTAL_ROWS,
    "columns": REFERENCE_COLUMNS,
    "dtypes": REFERENCE_DTYPES,
    "column_description": inv.describe_columns(ref).to_dict(orient="records"),
    "has_timestamp": HAS_TIMESTAMP,
    "has_endpoint_identifiers": HAS_ENDPOINTS,
    "timestamp_search": {"keywords": inv.TIME_KEYWORDS, **time_hits},
    "endpoint_search": {"keywords": inv.ENDPOINT_KEYWORDS, **endpoint_hits},
    "schema_identical_across_files": SCHEMA_IDENTICAL,
    "files_differing_from_reference": column_differences,
    "tiers": {"A": tier_a, "B": tier_b},
    "classes": {
        row["label"]: {
            "tier": row["tier"],
            "group6": row["group6"],
            "group2": row["group2"],
            "rows": int(row["rows"]),
            "pct_of_total": float(row["pct_of_total"]),
            "n_train_captures": int(row["n_train_captures"]),
            "n_test_captures": int(row["n_test_captures"]),
            "captures_train": row["captures_train"],
            "captures_test": row["captures_test"],
        }
        for _, row in class_sizes.iterrows()
    },
    "rows_per_file": {name: int(n) for name, n in ROW_COUNTS.items()},
    "rows_per_group6": {k: int(v) for k, v in six["rows"].items()},
    "rows_per_group2": {k: int(v) for k, v in two["rows"].items()},
    "imbalance_ratio": {
        "19_class": round(RATIO_19, 3),
        "6_class": round(RATIO_6, 3),
        "2_class": round(RATIO_2, 3),
    },
    "constant_columns": {
        "unit": "one file",
        "constant_everywhere": DEAD_COLUMNS,
        "recording_identifying": IDENTIFYING,
        "constant_in_some_recordings_only": near_miss["column"].tolist(),
        "grouping_by_recording_name_agrees": bool(GROUPING_AGREES),
        "per_column": [
            {
                "column": row["column"],
                "constant_in_recordings": int(row["constant_in_recordings"]),
                "of_recordings": int(row["of_recordings"]),
                "distinct_values_across_recordings": int(row["distinct_values_across_recordings"]),
                "constant_everywhere": bool(row["constant_everywhere"]),
                "recording_identifying": bool(row["recording_identifying"]),
                "values": row["values"][:20],
            }
            for _, row in report.iterrows()
        ],
    },
    "figures": [FIG_SIZES.name, FIG_RECORDINGS.name],
}

INVENTORY_PATH = OUT_DIR / "dataset_inventory.json"
INVENTORY_TEXT = json.dumps(cap.jsonable(inventory_doc), indent=2, default=str) + "\n"
INVENTORY_PATH.write_text(INVENTORY_TEXT)
print(f"wrote {INVENTORY_PATH}   ({len(INVENTORY_TEXT):,} bytes)")

The last thing to write is a first grouping of the columns into timing, protocol,
statistical and other. Later notebooks need to be able to ask what a model scores using
only the timing columns, or only the protocol ones, and that question needs the groups
fixed in a file rather than retyped each time.

This grouping is guesswork from spelling and nothing else. No column was opened to see
what it actually measures, so the file records itself as a draft and lists every column
that matched more than one rule. Rate is the obvious problem: it reads as timing to me
because it is a per-second quantity, but it is computed from a count, and a name like
`ack_count` has an equal claim on protocol and on statistical. I would rather flag those
than quietly decide them.

It gets written to the artefacts folder, which is the copy that survives. The copy inside
the cloned repository goes when the session does, so the file has to be moved into
`config/` from the Mac afterwards.

In [ ]:
assignment = inv.assign_families(REFERENCE_COLUMNS)

for name, cols in assignment["families"].items():
    print(f"{name:<12} {len(cols):>3}")
    for col in cols:
        print(f"             {col}")
print()
print(f"label columns : {assignment['labels'] or 'none found by name'}")
print(f"ambiguous     : {len(assignment['ambiguous'])} column(s) matched more than one rule")
for col, names in assignment["ambiguous"].items():
    print(f"                {col}  ->  {' / '.join(names)}   (assigned to {names[0]})")

FAMILIES_TEXT = inv.render_feature_families_yaml(
    assignment,
    source_file=REFERENCE_FILE.name,
    git_sha=GIT_SHA,
    run_date=RUN_DATE,
    generated_by="AG_PRAXIS_NB01_dataset_inventory.ipynb",
)

FAMILIES_PATH = OUT_DIR / "feature_families.yaml"
FAMILIES_PATH.write_text(FAMILIES_TEXT)
print()
print(f"wrote {FAMILIES_PATH} as a draft")

repo_copy = REPO_ROOT / "config" / "feature_families.yaml"
repo_copy.write_text(FAMILIES_TEXT)
print(f"wrote {repo_copy}")
if IN_COLAB:
    print("The repository copy goes when the session does. Move the Drive copy into")
    print("config/feature_families.yaml from the Mac and commit it there.")

Both files get printed in full here. Colab cannot push to the repository from a cell, so
until they are moved across by hand the only durable record of what this run produced is
the saved copy of this notebook. Printing them means the executed notebook carries the
whole result even if the artefacts folder is later cleared.

In [ ]:
for path in (INVENTORY_PATH, FAMILIES_PATH):
    print("=" * 79)
    print(f"{path}   ({path.stat().st_size:,} bytes)")
    print("=" * 79)
    print(path.read_text().rstrip())
    print()

print("=" * 79)
print("figures, which cannot be printed")
print("=" * 79)
for path in sorted(OUT_DIR.glob("*.png")):
    print(f"{path.name}   {path.stat().st_size:,} bytes")

The entry for `RESULTS_LEDGER.md`, ready to paste.

In [ ]:
status = "reference run" if SCAN_FULL else "EXPLORATORY, FAST=1, constancy check is a screen only"

ledger = f'''
### NB01 — dataset inventory ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB01_dataset_inventory.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| FAST | {int(FAST)} |
| status | {status} |
| files read | {len(ALL_FILES)} of {EXPECTED_FILES} expected |
| columns | {len(REFERENCE_COLUMNS)} |
| same columns in every file | {"yes" if SCHEMA_IDENTICAL else f"no, {len(column_differences)} file(s) differ"} |
| timestamp column | {"yes" if HAS_TIMESTAMP else "no"} |
| endpoint identifiers | {"yes" if HAS_ENDPOINTS else "no"} |
| classes | {N_CLASSES} |
| recordings | {N_CAPTURES} |
| tier A / tier B | {len(tier_a)} / {len(tier_b)} |
| total rows | {TOTAL_ROWS:,} |
| rows scanned for constancy | {ROWS_SCANNED:,} ({100 * ROWS_SCANNED / TOTAL_ROWS:.1f}%) |
| largest class | {largest["label"]}, {int(largest["rows"]):,} rows, {largest["pct_of_total"]}% |
| smallest class | {smallest["label"]}, {int(smallest["rows"]):,} rows, {smallest["pct_of_total"]}% |
| imbalance, 19-class | {RATIO_19:,.1f} to 1 |
| imbalance, 6-class | {RATIO_6:,.1f} to 1 |
| imbalance, 2-class | {RATIO_2:,.1f} to 1 |
| columns constant everywhere | {len(DEAD_COLUMNS)}{": " + ", ".join(DEAD_COLUMNS) if DEAD_COLUMNS else ""} |
| columns naming the recording | {len(IDENTIFYING)}{": " + ", ".join(IDENTIFYING) if IDENTIFYING else ""} |
| metrics | none, no model trained |
| artefacts | {OUT_DIR}, holding dataset_inventory.json, feature_families.yaml (draft), and two figures |

Tier B classes, which admit no capture-disjoint split: {", ".join(tier_b)}
'''

print(ledger)